# Millionare Client

In [35]:
%pip install -U bitsandbytes accelerate transformers

Note: you may need to restart the kernel to use updated packages.


## Models.py

In [36]:
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Any
from datetime import datetime
from enum import Enum

class GameStatus(Enum):
    """Possible game session statuses."""
    IN_PROGRESS = "in_progress"
    COMPLETED = "completed"
    FAILED = "failed"
    TIMEOUT = "timeout"


@dataclass
class User:
    """Represents a user account."""
    id: int
    username: str
    role: str

    @classmethod
    def from_dict(cls, data: dict) -> "User":
        return cls(
            id=data["id"],
            username=data["username"],
            role=data["role"]
        )


@dataclass
class Option:
    """Represents an answer option for a question."""
    id: int
    text: Optional[str] = None

    @classmethod
    def from_dict(cls, data: dict) -> "Option":
        return cls(
            id=data["id"],
            text=data.get("text")
        )


@dataclass
class Question:
    """Represents a quiz question."""
    id: int
    text: Optional[str] = None
    options: List[Option] = field(default_factory=list)
    level: int = 0

    @classmethod
    def from_dict(cls, data: dict) -> "Question":
        return cls(
            id=data["id"],
            text=data.get("text"),
            options=[Option.from_dict(opt) for opt in data.get("options", [])],
            level=data.get("level", 0)
        )

    def get_option_by_id(self, option_id: int) -> Optional[Option]:
        """Get an option by its ID."""
        for opt in self.options:
            if opt.id == option_id:
                return opt
        return None

    def get_option_by_text(self, text: str, case_sensitive: bool = False) -> Optional[Option]:
        """Get an option by its text content."""
        for opt in self.options:
            if opt.text is None:
                continue
            opt_text = opt.text if case_sensitive else opt.text.lower()
            search_text = text if case_sensitive else text.lower()
            if opt_text == search_text:
                return opt
        return None


@dataclass
class MoneyLevel:
    """Represents a prize level in the money pyramid."""
    level: int
    amount: float

    @classmethod
    def from_dict(cls, data: dict) -> "MoneyLevel":
        return cls(
            level=data["level"],
            amount=data["amount"]
        )


@dataclass
class Competition:
    """Represents a game competition."""
    id: int
    name: str
    description: Optional[str] = None
    max_levels: int = 15
    is_infinite: bool = False
    created_at: Optional[str] = None
    question_count: Optional[int] = None

    @classmethod
    def from_dict(cls, data: dict) -> "Competition":
        return cls(
            id=data["id"],
            name=data["name"],
            description=data.get("description"),
            max_levels=data.get("maxLevels", 15),
            is_infinite=data.get("isInfinite", False),
            created_at=data.get("createdAt"),
            question_count=data.get("questionCount")
        )


@dataclass
class CompetitionConfig:
    """Represents prize details for a competition."""
    money_pyramid: List[MoneyLevel]

    @classmethod
    def from_dict(cls, data: dict) -> "CompetitionConfig":
        return cls(
            money_pyramid=[MoneyLevel.from_dict(ml) for ml in data.get("moneyPyramid", [])]
        )


@dataclass
class LeaderboardEntry:
    """Represents a row inside a leaderboard."""
    rank: int
    username: str
    score: float
    level: int
    date: str

    @classmethod
    def from_dict(cls, data: dict) -> "LeaderboardEntry":
        return cls(
            rank=data["rank"],
            username=data["username"],
            score=data["score"],
            level=data["level"],
            date=data["date"]
        )


@dataclass
class Leaderboard:
    """Represents a complete leaderboard structure."""
    entries: List[LeaderboardEntry]

    @classmethod
    def from_dict(cls, data: dict) -> "Leaderboard":
        return cls(
            entries=[LeaderboardEntry.from_dict(entry) for entry in data.get("entries", [])]
        )


@dataclass
class GameState:
    """Represents the complete state of a game session."""
    session_id: int
    competition: Competition
    status: GameStatus
    earned_amount: float
    current_level: int
    money_pyramid: List[MoneyLevel]
    mode: str
    time_remaining: Optional[float] = None
    question: Optional[Question] = None

    @classmethod
    def from_dict(cls, data: dict) -> "GameState":
        # 1. Safe fallback for the status field
        if "status" in data:
            status_str = data["status"].upper() if isinstance(data["status"], str) else data["status"].name
        else:
            status_str = "IN_PROGRESS" if data.get("question") is not None else "COMPLETED"

        # 2. Reconstruct the Question object manually
        current_question = None
        question_data = data.get("question")
        
        if question_data:
            options_list = []
            for opt in question_data.get("options", []):
                options_list.append(AnswerOption(opt["id"], opt["text"]))
                
            current_question = Question(
                question_data["id"], 
                question_data["text"], 
                options_list
            )

        # 3. Build and return the final GameState object using the exact positional order
        # Order required by your __init__: session_id, status, earned_amount, current_level, money_pyramid, mode, question
        return cls(
            data["sessionId"],
            status_str,
            data["earnedAmount"],
            data["currentLevel"],
            data["moneyPyramid"],
            data["mode"],
            current_question
        )

    @property
    def in_progress(self) -> bool:
        return self.status == GameStatus.IN_PROGRESS

    @property
    def is_game_over(self) -> bool:
        return self.status in (GameStatus.COMPLETED, GameStatus.FAILED, GameStatus.TIMEOUT)


@dataclass
class AnswerResult:
    """Represents the result returned by the server after answering."""
    correct: Optional[bool]
    earned_amount: float
    game_over: bool
    status: str
    current_level: Optional[int] = None
    question: Optional[Question] = None
    money_pyramid: Optional[List[MoneyLevel]] = None
    question_deadline: Optional[datetime] = None
    timed_out: bool = False

    @classmethod
    def from_dict(cls, data: dict) -> "AnswerResult":
        deadline_str = data.get("questionDeadline")
        deadline = datetime.fromisoformat(deadline_str) if deadline_str else None
        
        return cls(
            correct=data.get("correct"),
            earned_amount=data["earnedAmount"],
            game_over=data["gameOver"],
            status=data["status"],
            current_level=data.get("currentLevel"),
            question=Question.from_dict(data["question"]) if data.get("question") else None,
            money_pyramid=[MoneyLevel.from_dict(ml) for ml in data["moneyPyramid"]] if data.get("moneyPyramid") else None,
            question_deadline=deadline,
            timed_out=data.get("status") == "timeout"
        )

## Exceptions.py

In [37]:
class MillionaireError(Exception):
    """Base exception for all Millionaire client errors."""
    def __init__(self, message: str, status_code: Optional[int] = None, response_data: Optional[dict] = None):
        super().__init__(message)
        self.message = message
        self.status_code = status_code
        self.response_data = response_data or {}

    def __str__(self):
        if self.status_code:
            return f"[{self.status_code}] {self.message}"
        return self.message

class AuthenticationError(MillionaireError): pass
class GameError(MillionaireError): pass
class TimeoutError(MillionaireError): pass
class ValidationError(MillionaireError): pass
class NotFoundError(MillionaireError): pass
class ServerError(MillionaireError): pass
class RateLimitError(MillionaireError): pass

## base.py

In [38]:
import requests
from urllib.parse import urljoin

class BaseClient:
    """Base HTTP client handling authentication and request/response logic."""
    def __init__(self, base_url: str, timeout: int = 30):
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout
        self._session = requests.Session()
        self._auth_cookie: Optional[str] = None
    
    def _get_full_url(self, endpoint: str) -> str:
        return urljoin(f"{self.base_url}/", endpoint.lstrip("/"))
    
    def set_auth_cookie(self, cookie_value: str):
        self._auth_cookie = cookie_value
        self._session.cookies.set("polimillionaire_auth", cookie_value)
    
    def clear_auth(self):
        self._auth_cookie = None
        self._session.cookies.clear()
    
    @property
    def is_authenticated(self) -> bool:
        return "polimillionaire_auth" in self._session.cookies
    
    def _handle_response(self, response: requests.Response) -> Dict[str, Any]:
        try:
            data = response.json() if response.text else {}
        except ValueError:
            data = {}

        if response.status_code in (200, 201, 204):
            return data
        
        message = data.get("message", data.get("error", f"HTTP {response.status_code}"))

        if response.status_code == 401:
            raise AuthenticationError(message, response.status_code, data)
        elif response.status_code == 404:
            raise NotFoundError(message, response.status_code, data)
        elif response.status_code == 400:
            raise ValidationError(message, response.status_code, data)
        elif response.status_code == 429:
            raise RateLimitError(message, response.status_code, data)
        elif response.status_code >= 500:
            raise ServerError(message, response.status_code, data)
        else:
            raise MillionaireError(message, response.status_code, data)
    
    def request(self, method: str, endpoint: str, data: Optional[Dict] = None, params: Optional[Dict] = None, headers: Optional[Dict] = None, auth_required: bool = True, raw: bool = False) -> Any:
        if auth_required and not self.is_authenticated:
            raise AuthenticationError("Authentication required. Call login() first.")
        
        url = self._get_full_url(endpoint)
        request_headers = headers or {}
        
        try:
            if method.upper() == "GET":
                response = self._session.get(url, params=params, headers=request_headers, timeout=self.timeout)
            elif method.upper() == "POST":
                response = self._session.post(url, json=data, params=params, headers=request_headers, timeout=self.timeout)
            elif method.upper() == "PUT":
                response = self._session.put(url, json=data, headers=request_headers, timeout=self.timeout)
            elif method.upper() == "PATCH":
                response = self._session.patch(url, json=data, headers=request_headers, timeout=self.timeout)
            elif method.upper() == "DELETE":
                response = self._session.delete(url, headers=request_headers, timeout=self.timeout)
            else:
                raise ValueError(f"Unsupported HTTP method: {method}")
            
            if raw and response.status_code in (200, 201, 204):
                return response.content
            return self._handle_response(response)
            
        except requests.Timeout:
            raise MillionaireError(f"Request to {endpoint} timed out after {self.timeout}s")
        except requests.ConnectionError as e:
            raise MillionaireError(f"Could not connect to server at {self.base_url}: {e}")
    
    def get(self, endpoint: str, **kwargs) -> Any: return self.request("GET", endpoint, **kwargs)
    def post(self, endpoint: str, **kwargs) -> Any: return self.request("POST", endpoint, **kwargs)
    def put(self, endpoint: str, **kwargs) -> Any: return self.request("PUT", endpoint, **kwargs)
    def patch(self, endpoint: str, **kwargs) -> Any: return self.request("PATCH", endpoint, **kwargs)
    def delete(self, endpoint: str, **kwargs) -> Any: return self.request("DELETE", endpoint, **kwargs)

## auth.py

In [39]:
class AuthModule:
    """Handles authentication operations."""
    def __init__(self, client: BaseClient):
        self._client = client
    
    def login(self, username: str, password: str) -> User:
        response = self._client.post(
            "/api/auth/login",
            data={"username": username, "password": password},
            auth_required=False
        )
        return User.from_dict(response["user"])
    
    def logout(self):
        try:
            self._client.post("/api/auth/logout", auth_required=False)
        finally:
            self._client.clear_auth()
    
    def get_current_user(self) -> User:
        response = self._client.get("/api/auth/me")
        return User.from_dict(response["user"])
    
    def is_logged_in(self) -> bool:
        return self._client.is_authenticated

## competitions.py

In [40]:
class CompetitionsModule:
    """Handles competition-related operations."""
    def __init__(self, client: BaseClient):
        self._client = client

    def list_all(self) -> List[Competition]:
        response = self._client.get("/api/competitions")
        return [Competition.from_dict(c) for c in response.get("competitions", [])]

    def get_config(self, competition_id: int) -> CompetitionConfig:
        response = self._client.get(f"/api/competitions/{competition_id}/config")
        return CompetitionConfig.from_dict(response)

    def find_by_name(self, name: str, case_sensitive: bool = False) -> Competition:
        competitions = self.list_all()
        search_name = name if case_sensitive else name.lower()
        for comp in competitions:
            comp_name = comp.name if case_sensitive else comp.name.lower()
            if comp_name == search_name:
                return comp
        available = [c.name for c in competitions]
        raise ValueError(f"Competition '{name}' not found. Available: {available}")

## Leaderboard.py

In [41]:
class LeaderboardModule:
    """Handles leaderboard operations."""
    def __init__(self, client: BaseClient):
        self._client = client
    
    def get(self, competition_id: int, limit: int = 10, mode: str = "text") -> Leaderboard:
        response = self._client.get(
            f"/api/leaderboard/{competition_id}",
            params={"limit": min(max(limit, 1), 100), "mode": mode}
        )
        return Leaderboard.from_dict(response)
    
    def get_top(self, competition_id: int, n: int = 10) -> List[LeaderboardEntry]:
        return self.get(competition_id, limit=n).entries[:n]
    
    def find_player(self, competition_id: int, username: str, mode: str = "text") -> Optional[LeaderboardEntry]:
        leaderboard = self.get(competition_id, limit=100, mode=mode)
        for entry in leaderboard.entries:
            if entry.username.lower() == username.lower():
                return entry
        return None

## game.py

In [42]:
class GameSession:
    """Represents an active game session."""
    def __init__(self, client: BaseClient, state: GameState):
        self._client = client
        self._state = state
    
    @property
    def session_id(self) -> int: return self._state.session_id
    @property
    def state(self) -> GameState: return self._state
    @property
    def current_question(self) -> Optional[Question]: return self._state.question
    @property
    def current_level(self) -> int: return self._state.current_level
    @property
    def earned_amount(self) -> float: return self._state.earned_amount
    @property
    def in_progress(self) -> bool: return self._state.in_progress
    @property
    def is_game_over(self) -> bool: return self._state.is_game_over
    @property
    def time_remaining(self) -> Optional[float]: return self._state.time_remaining
    @property
    def money_pyramid(self) -> List: return self._state.money_pyramid
    @property
    def mode(self) -> str: return self._state.mode
    
    def fetch_audio_question(self) -> bytes:
        if self.mode != "speech": raise GameError("Audio is only available in speech mode")
        return self._client.get(f"/api/game/{self.session_id}/audio/question", raw=True)
    
    def fetch_audio_option_next(self) -> bytes:
        if self.mode != "speech": raise GameError("Audio is only available in speech mode")
        return self._client.get(f"/api/game/{self.session_id}/audio/option/next", raw=True)
    
    def fetch_audio_option(self, index: int) -> bytes:
        if self.mode != "speech": raise GameError("Audio is only available in speech mode")
        if not 0 <= index <= 3: raise ValueError("Option index must be 0, 1, 2, or 3")
        return self._client.get(f"/api/game/{self.session_id}/audio/option/{index}", raw=True)
    
    def refresh_state(self) -> GameState:
        response = self._client.get(f"/api/game/{self.session_id}/state")
        self._state = GameState.from_dict(response)
        return self._state
    
    def answer(self, option_id: int) -> AnswerResult:
        if not self._state.question:
            raise GameError("No active question to answer")
        try:
            response = self._client.post(f"/api/game/{self.session_id}/answer", data={"optionId": option_id})
        except Exception as e:
            if "timeout" in str(e).lower(): raise TimeoutError("Question timed out")
            raise
        
        result = AnswerResult.from_dict(response)
        if result.timed_out or result.status == "timeout":
            self._state.status = GameStatus.TIMEOUT
            self._state.earned_amount = result.earned_amount
            self._state.question = None
            return result

        if result.question:
            self._state = GameState.from_dict({
                "sessionId": self.session_id,
                "competition": self._state.competition.__dict__,
                "status": "in_progress" if not result.game_over else "completed",
                "earnedAmount": result.earned_amount,
                "currentLevel": result.current_level or self._state.current_level,
                "moneyPyramid": [ml.__dict__ for ml in result.money_pyramid] if result.money_pyramid else self._state.money_pyramid,
                "questionDeadline": result.question_deadline.isoformat() if result.question_deadline else None,
                "mode": self._state.mode,
                "question": {
                    "id": result.question.id,
                    "level": result.question.level,
                    "text": result.question.text,
                    "options": [{"id": opt.id, "text": opt.text} for opt in result.question.options]
                } if result.question else None
            })
        elif result.game_over:
            if result.correct: self._state.status = GameStatus.COMPLETED
            elif result.correct is False: self._state.status = GameStatus.FAILED
            self._state.earned_amount = result.earned_amount
            self._state.question = None
        return result
    
    def answer_by_text(self, answer_text: str, case_sensitive: bool = False) -> AnswerResult:
        if not self._state.question: raise GameError("No active question to answer")
        option = self._state.question.get_option_by_text(answer_text, case_sensitive)
        if not option:
            available = [opt.text for opt in self._state.question.options]
            raise GameError(f"Answer '{answer_text}' not found. Available: {available}")
        return self.answer(option.id)
    
    def timeout(self):
        return self._client.post(f"/api/game/{self.session_id}/timeout")


class GameModule:
    """Handles game-related operations."""
    def __init__(self, client: BaseClient):
        self._client = client
    
    def start(self, competition_id: int, mode: str = "text") -> GameSession:
        response = self._client.post("/api/game/start", data={"competitionId": competition_id, "mode": mode})
        print("DEBUG: Server response raw data ->", response)
        state = GameState.from_dict(response)
        return GameSession(self._client, state)
    
    def get_state(self, session_id: int) -> GameState:
        response = self._client.get(f"/api/game/{session_id}/state")
        return GameState.from_dict(response)

## client.py

In [43]:
class MillionaireClient:
    """Main client for interacting with the Poli-Millionaire API."""
    def __init__(self, base_url: str, timeout: int = 30):
        self._base = BaseClient(base_url, timeout)
        self._auth = AuthModule(self._base)
        self._game = GameModule(self._base)
        self._competitions = CompetitionsModule(self._base)
        self._leaderboard = LeaderboardModule(self._base)
    
    def login(self, username: str, password: str) -> User: return self._auth.login(username, password)
    def logout(self): self._auth.logout()
    
    @property
    def user(self) -> User: return self._auth.get_current_user()
    @property
    def is_authenticated(self) -> bool: return self._auth.is_logged_in()
    @property
    def auth(self) -> AuthModule: return self._auth
    @property
    def game(self) -> GameModule: return self._game
    @property
    def competitions(self) -> CompetitionsModule: return self._competitions
    @property
    def leaderboard(self) -> LeaderboardModule: return self._leaderboard
    
    def play_game(self, competition_id: int, answer_strategy, mode: str = "text"):
        game = self._game.start(competition_id, mode=mode)
        while game.in_progress:
            question = game.current_question
            if not question: break
            
            answer = answer_strategy(question)
            if isinstance(answer, int):
                result = game.answer(answer)
            else:
                result = game.answer_by_text(str(answer))
            
            if result.game_over: break
        return game

## proof

In [44]:
# Instanciar el cliente integrado en 

from kaggle_secrets import UserSecretsClient
rey_user = UserSecretsClient().get_secret("rey_user")
rey_password = UserSecretsClient().get_secret("rey_password")

client = MillionaireClient("http://131.175.15.22:51111")
client.login(rey_user, rey_password)

User(id=74, username='reyci', role='student')

# src/polimillionare

## types.py

In [45]:
from __future__ import annotations

"""Small dataclasses shared by the assignment notebook helpers."""

from dataclasses import dataclass, field
from typing import Any


@dataclass(frozen=True)
class AnswerOption:
    id: int
    text: str


@dataclass(frozen=True)
class Question:
    id: int
    text: str
    options: list[AnswerOption]
    level: int = 0
    metadata: dict[str, Any] = field(default_factory=dict)

    def valid_option_ids(self) -> set[int]:
        return {option.id for option in self.options}

    def first_option(self) -> AnswerOption:
        if not self.options:
            raise ValueError("Question has no answer options")
        return self.options[0]

    def get_option(self, option_id: int) -> AnswerOption | None:
        for option in self.options:
            if option.id == option_id:
                return option
        return None

    def require_option(self, option_id: int) -> AnswerOption:
        return self.get_option(option_id) or self.first_option()


@dataclass
class AnswerPrediction:
    option_id: int
    answer_text: str
    confidence: float | None = None
    reasoning: str | None = None
    metadata: dict[str, Any] = field(default_factory=dict)


## strategies.py

In [46]:
from __future__ import annotations

"""Strategies, local model loading, prompting, and output parsing."""

import json
import random
import re
from abc import ABC, abstractmethod
from dataclasses import dataclass, fields
from typing import Any, Protocol
from langchain_core.tools import tool, render_text_description
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from typing import Dict, Any, Optional

from packaging.version import Version



class BaseStrategy(ABC):
    name = "base"

    @abstractmethod
    def answer(self, question: Question) -> AnswerPrediction:
        raise NotImplementedError


class RandomStrategy(BaseStrategy):
    name = "random"

    def __init__(self, seed: int | None = None):
        self._rng = random.Random(seed)

    def answer(self, question: Question) -> AnswerPrediction:
        option = self._rng.choice(question.options)
        return AnswerPrediction(
            option_id=option.id,
            answer_text=option.text,
            confidence=1.0 / max(1, len(question.options)),
            reasoning="Random option selected.",
            metadata={"strategy": self.name},
        )


class HeuristicStrategy(BaseStrategy):
    name = "heuristic"

    def answer(self, question: Question) -> AnswerPrediction:
        question_terms = set(_words(question.text))
        option = max(
            question.options,
            key=lambda item: len(question_terms & set(_words(item.text))),
        )
        return AnswerPrediction(
            option_id=option.id,
            answer_text=option.text,
            confidence=0.35,
            reasoning="Simple word-overlap heuristic selected the option.",
            metadata={"strategy": self.name},
        )


class LocalLLM(Protocol):
    model_name: str

    def generate(self, prompt: str, **kwargs: object) -> str:
        ...


class FakeLLM:
    def __init__(self, responses: list[str] | None = None, model_name: str = "fake-llm"):
        self.responses = list(responses or ['{"option_id": 0, "confidence": 0.5, "reason": "fake"}'])
        self.model_name = model_name
        self.prompts: list[str] = []
        self.calls: list[dict[str, object]] = []

    def generate(self, prompt: str, **kwargs: object) -> str:
        self.prompts.append(prompt)
        self.calls.append(dict(kwargs))
        if len(self.responses) == 1:
            return self.responses[0]
        return self.responses.pop(0)


@dataclass
class GemmaLLMConfig:
    model_id: str = "google/gemma-4-E2B-it"
    inference_backend: str = "auto_model"
    device_map: str = "auto"
    dtype: str = "auto"
    max_new_tokens: int = 8
    temperature: float = 0.0
    do_sample: bool = False
    num_beams: int = 1
    seed: int | None = 42
    generation_max_time_seconds: float | None = 18.0
    timeout_seconds: float | None = 25.0
    quantize_4bit: bool = False


@dataclass
class QwenLLMConfig:
    model_id: str = "Qwen/Qwen3.5-2B"
    device_map: str = "auto"
    dtype: str = "auto"
    max_new_tokens: int = 256
    temperature: float = 1.0
    do_sample: bool = True
    top_p: float = 0.95
    top_k: int = 20
    seed: int | None = 42
    enable_thinking: bool = True
    generation_max_time_seconds: float | None = None
    timeout_seconds: float | None = 25.0
    quantize_4bit: bool = False


class GemmaLLM:
    def __init__(self, config: GemmaLLMConfig | None = None, **overrides: Any):
        self.config = config or GemmaLLMConfig()
        for key, value in overrides.items():
            if hasattr(self.config, key):
                setattr(self.config, key, value)
        self.model_name = self.config.model_id
        self._model: Any = None
        self._processor: Any = None
        self._tokenizer: Any = None
        self._pipeline: Any = None

    @property
    def is_loaded(self) -> bool:
        return self._model is not None or self._pipeline is not None

    @property
    def device_summary(self) -> str:
        if self._model is not None:
            device_map = getattr(self._model, "hf_device_map", None)
            if device_map:
                return str(device_map)
            try:
                return str(next(self._model.parameters()).device)
            except Exception:
                return "unknown"
        if self._pipeline is not None:
            return str(getattr(self._pipeline, "device", "pipeline"))
        return "not_loaded"

    def generate(self, prompt: str, **kwargs: Any) -> str:
        self._load()
        seed = kwargs.pop("seed", self.config.seed)
        self._seed(seed)
        generation_kwargs = self._generation_kwargs(kwargs)
        if self.config.inference_backend == "auto_model":
            text = self._generate_auto_model(prompt, generation_kwargs)
        elif self.config.inference_backend == "pipeline_any_to_any":
            text = self._generate_pipeline(prompt, generation_kwargs)
        else:
            raise ValueError(f"Unsupported inference_backend: {self.config.inference_backend}")
        return text.strip()
    
    def invoke(self, input_data: Any, config: Any = None) -> Any:
        from langchain_core.outputs import ChatResult, ChatGeneration
        from langchain_core.messages import BaseMessage
        
        if hasattr(input_data, "to_string"):
            prompt_text = input_data.to_string()
        elif isinstance(input_data, list) and len(input_data) > 0:
            prompt_text = getattr(input_data[-1], "content", str(input_data))
        else:
            prompt_text = str(input_data)
            

        generated_text = self.generate(prompt_text)
        

        return str(generated_text)

    def _load(self) -> None:
        if self.is_loaded:
            return
        if self.config.inference_backend == "auto_model":
            self._load_auto_model()
        elif self.config.inference_backend == "pipeline_any_to_any":
            self._load_pipeline()
        else:
            raise ValueError(f"Unsupported inference_backend: {self.config.inference_backend}")

    def _load_auto_model(self) -> None:
        from transformers import AutoModelForCausalLM, AutoProcessor, AutoTokenizer
        import transformers

        _require_supported_transformers(transformers.__version__)
        try:
            self._processor = AutoProcessor.from_pretrained(self.config.model_id)
        except Exception:
            self._tokenizer = AutoTokenizer.from_pretrained(self.config.model_id, extra_special_tokens={})
        self._model = AutoModelForCausalLM.from_pretrained(
            self.config.model_id,
            device_map=self.config.device_map,
            dtype=self.config.dtype,
            **_quantization_kwargs(self.config.quantize_4bit),
        )
        self._model.eval()

    def _load_pipeline(self) -> None:
        from transformers import pipeline

        kwargs: dict[str, Any] = {
            "task": "any-to-any",
            "model": self.config.model_id,
            "device_map": self.config.device_map,
            "dtype": self.config.dtype,
        }
        if self.config.quantize_4bit:
            kwargs["model_kwargs"] = _quantization_kwargs(True)
        self._pipeline = pipeline(**kwargs)

    def _generation_kwargs(self, overrides: dict[str, Any]) -> dict[str, Any]:
        kwargs = {
            "max_new_tokens": self.config.max_new_tokens,
            "temperature": self.config.temperature,
            "do_sample": self.config.do_sample,
            "num_beams": self.config.num_beams,
        }
        if self.config.generation_max_time_seconds is not None:
            kwargs["max_time"] = self.config.generation_max_time_seconds
        kwargs.update({key: value for key, value in overrides.items() if value is not None})
        if not kwargs["do_sample"]:
            kwargs.pop("temperature", None)
        return kwargs

    def _seed(self, seed: int | None) -> None:
        if seed is None:
            return
        try:
            import torch

            torch.manual_seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed)
        except Exception:
            pass

    def _generate_auto_model(self, prompt: str, generation_kwargs: dict[str, Any]) -> str:
        import torch

        processor = self._processor or self._tokenizer
        inputs = _tokenize_prompt(processor, prompt)
        try:
            device = next(self._model.parameters()).device
            inputs = {key: value.to(device) if hasattr(value, "to") else value for key, value in inputs.items()}
        except Exception:
            pass
        input_length = inputs["input_ids"].shape[-1]
        with torch.inference_mode():
            output_ids = self._model.generate(**inputs, **generation_kwargs)
        generated_ids = output_ids[0][input_length:]
        return processor.decode(generated_ids, skip_special_tokens=True)

    def _generate_pipeline(self, prompt: str, generation_kwargs: dict[str, Any]) -> str:
        messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
        result = self._pipeline(messages, return_full_text=False, generate_kwargs=generation_kwargs)
        if isinstance(result, list) and result:
            item = result[0]
            if isinstance(item, dict):
                return str(item.get("generated_text", ""))
            return str(item)
        return str(result)


class QwenLLM:
    def __init__(self, config: QwenLLMConfig | None = None, **overrides: Any):
        self.config = config or QwenLLMConfig()
        for key, value in overrides.items():
            if hasattr(self.config, key):
                setattr(self.config, key, value)
        self.model_name = self.config.model_id
        self._model: Any = None
        self._processor: Any = None

    @property
    def is_loaded(self) -> bool:
        return self._model is not None

    @property
    def device_summary(self) -> str:
        if self._model is None:
            return "not_loaded"
        device_map = getattr(self._model, "hf_device_map", None)
        if device_map:
            return str(device_map)
        try:
            return str(next(self._model.parameters()).device)
        except Exception:
            return "unknown"

    def generate(self, prompt: str, **kwargs: Any) -> str:
        self._load()
        seed = kwargs.pop("seed", self.config.seed)
        self._seed(seed)
        generation_kwargs = self._generation_kwargs(kwargs)
        messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
        inputs = self._processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            enable_thinking=self.config.enable_thinking,
        )
        try:
            device = next(self._model.parameters()).device
            inputs = {key: value.to(device) if hasattr(value, "to") else value for key, value in inputs.items()}
        except Exception:
            pass

        import torch

        input_length = inputs["input_ids"].shape[-1]
        with torch.inference_mode():
            output_ids = self._model.generate(**inputs, **generation_kwargs)
        generated_ids = output_ids[0][input_length:]
        return self._processor.decode(generated_ids, skip_special_tokens=True).strip()
    
    def invoke(self, input_data: Any, config: Any = None) -> Any:
    
        from langchain_core.outputs import ChatResult, ChatGeneration
        from langchain_core.messages import BaseMessage
        
    
        if hasattr(input_data, "to_string"):
            prompt_text = input_data.to_string()
        elif isinstance(input_data, list) and len(input_data) > 0:
            prompt_text = getattr(input_data[-1], "content", str(input_data))
        else:
            prompt_text = str(input_data)
            
        generated_text = self.generate(prompt_text)
        
        class CustomContent:
            def __init__(self, text):
                self.content = [{'text': text}]
                self.text_content = text
            @property
            def str_output(self):
                return self.text_content
            def __str__(self):
                return self.text_content
                g
        return CustomContent(generated_text)

    def _load(self) -> None:
        if self.is_loaded:
            return
        from transformers import AutoModelForImageTextToText, AutoProcessor
        import transformers

        _require_supported_transformers(transformers.__version__)
        self._processor = AutoProcessor.from_pretrained(self.config.model_id)
        self._model = AutoModelForImageTextToText.from_pretrained(
            self.config.model_id,
            device_map=self.config.device_map,
            dtype=self.config.dtype,
            **_quantization_kwargs(self.config.quantize_4bit),
        )
        self._model.eval()

    def _generation_kwargs(self, overrides: dict[str, Any]) -> dict[str, Any]:
        kwargs = {
            "max_new_tokens": self.config.max_new_tokens,
            "do_sample": self.config.do_sample,
        }
        if self.config.do_sample:
            kwargs.update(
                {
                    "temperature": self.config.temperature,
                    "top_p": self.config.top_p,
                    "top_k": self.config.top_k,
                }
            )
        if self.config.generation_max_time_seconds is not None:
            kwargs["max_time"] = self.config.generation_max_time_seconds
        kwargs.update({key: value for key, value in overrides.items() if value is not None})
        return kwargs

    def _seed(self, seed: int | None) -> None:
        if seed is None:
            return
        try:
            import torch

            torch.manual_seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed)
        except Exception:
            pass


class GemmaStrategy(BaseStrategy):
    name = "gemma"

    def __init__(self, llm: LocalLLM | None = None, model_config: GemmaLLMConfig | dict[str, Any] | None = None):
        if llm is None:
            if isinstance(model_config, dict):
                allowed = {field.name for field in fields(GemmaLLMConfig)}
                model_config = GemmaLLMConfig(**{key: value for key, value in model_config.items() if key in allowed})
            llm = GemmaLLM(model_config)
        self.llm = llm

    def answer(self, question: Question) -> AnswerPrediction:
        raw_text = self.llm.generate(build_prompt(question))
        prediction = parse_answer_prediction(raw_text, question, strategy_name=self.name)
        prediction.metadata["strategy"] = self.name
        prediction.metadata["model_name"] = getattr(self.llm, "model_name", "unknown")
        prediction.metadata["device"] = getattr(self.llm, "device_summary", "unknown")
        return prediction


class QwenStrategy(BaseStrategy):
    name = "qwen3.5_thinking"

    def __init__(self, llm: LocalLLM | None = None, model_config: QwenLLMConfig | dict[str, Any] | None = None):
        if isinstance(model_config, dict):
            allowed = {field.name for field in fields(QwenLLMConfig)}
            model_config = QwenLLMConfig(**{key: value for key, value in model_config.items() if key in allowed})
        self.config = model_config or QwenLLMConfig()
        self.llm = llm or QwenLLM(self.config)

    def answer(self, question: Question) -> AnswerPrediction:
        raw_text = self.llm.generate(build_qwen_prompt(question))
        prediction = parse_answer_prediction(raw_text, question, strategy_name=self.name)
        prediction.metadata["strategy"] = self.name
        prediction.metadata["model_name"] = getattr(self.llm, "model_name", "unknown")
        prediction.metadata["device"] = getattr(self.llm, "device_summary", "unknown")
        prediction.metadata["thinking"] = self.config.enable_thinking
        return prediction


class CouncilStrategy(BaseStrategy):
    name = "council"

    def __init__(
        self,
        llm: LocalLLM | None = None,
        judge_llm: LocalLLM | None = None,
        candidate_llms: list[LocalLLM] | None = None,
        num_votes: int = 3,
        base_seed: int = 100,
        candidate_temperature: float = 0.8,
        candidate_top_p: float = 0.9,
        candidate_max_new_tokens: int = 48,
        judge_max_new_tokens: int = 8,
        max_time_per_call: float | None = None,
        shuffle_options: bool = False,
        judge_scope: str = "any_option",
        rejected_judge_fallback: str = "confidence_weighted",
    ):
        if candidate_llms:
            self.candidate_llms = list(candidate_llms)
            self.llm = llm or self.candidate_llms[0]
        elif llm is not None:
            if num_votes < 1:
                raise ValueError("num_votes must be at least 1")
            self.llm = llm
            self.candidate_llms = [llm] * num_votes
        else:
            raise ValueError("Provide llm or candidate_llms")
        if not self.candidate_llms:
            raise ValueError("num_votes must be at least 1")
        self.judge_llm = judge_llm or self.llm
        self.num_votes = len(self.candidate_llms)
        self.base_seed = base_seed
        self.candidate_temperature = candidate_temperature
        self.candidate_top_p = candidate_top_p
        self.candidate_max_new_tokens = candidate_max_new_tokens
        self.judge_max_new_tokens = judge_max_new_tokens
        self.max_time_per_call = max_time_per_call
        self.shuffle_options = shuffle_options
        if judge_scope not in {"candidate_only", "any_option"}:
            raise ValueError("judge_scope must be 'candidate_only' or 'any_option'")
        self.judge_scope = judge_scope
        if rejected_judge_fallback not in {"confidence_weighted", "primary_candidate"}:
            raise ValueError("rejected_judge_fallback must be 'confidence_weighted' or 'primary_candidate'")
        self.rejected_judge_fallback = rejected_judge_fallback

    def answer(self, question: Question) -> AnswerPrediction:
        votes: list[AnswerPrediction] = []
        for vote_index, candidate_llm in enumerate(self.candidate_llms):
            option_order = self._option_order(question, vote_index)
            raw_text = candidate_llm.generate(
                build_council_vote_prompt(question, option_order),
                max_new_tokens=self.candidate_max_new_tokens,
                do_sample=True,
                temperature=self.candidate_temperature,
                top_p=self.candidate_top_p,
                seed=self.base_seed + vote_index,
                **self._time_kwargs(),
            )
            vote = parse_answer_prediction(raw_text, question, strategy_name="council_vote")
            if not vote.metadata["fallback"]:
                vote.metadata["model_name"] = getattr(candidate_llm, "model_name", "unknown")
                vote.metadata["sample_seed"] = self.base_seed + vote_index
                vote.metadata["option_order"] = [option.id for option in option_order]
                votes.append(vote)

        if not votes:
            return _council_fallback(question, "No valid candidate votes.", [])

        supported_options = {vote.option_id for vote in votes}
        majority_option = _majority_option(votes)
        if majority_option is not None:
            result = _selected_vote_prediction(question, votes, majority_option)
            method = "unanimous_vote" if len(supported_options) == 1 else "majority_vote"
            result.metadata.update(self._metadata(votes, None, method))
            return result

        judge_raw_text = self.judge_llm.generate(
            build_judge_prompt(question, votes, judge_scope=self.judge_scope),
            max_new_tokens=self.judge_max_new_tokens,
            do_sample=False,
            seed=self.base_seed + self.num_votes,
            **self._time_kwargs(),
        )
        judged = parse_answer_prediction(judge_raw_text, question, strategy_name=self.name)
        if not judged.metadata["fallback"] and (
            self.judge_scope == "any_option" or judged.option_id in supported_options
        ):
            judged.metadata.update(self._metadata(votes, judge_raw_text, "judge"))
            judged.metadata["judge_novel_choice"] = judged.option_id not in supported_options
            return judged

        if self.rejected_judge_fallback == "primary_candidate":
            result = _selected_vote_prediction(question, votes, votes[0].option_id)
            result.reasoning = "Primary candidate selected after judge returned an unsupported answer."
            method = "primary_candidate"
        else:
            result = _weighted_vote(question, votes)
            method = "weighted_vote"
        result.metadata.update(self._metadata(votes, judge_raw_text, method))
        result.metadata["judge_rejected"] = True
        result.metadata["judge_option_id"] = None if judged.metadata["fallback"] else judged.option_id
        return result

    def _time_kwargs(self) -> dict[str, float]:
        if self.max_time_per_call is None:
            return {}
        return {"max_time": self.max_time_per_call}

    def _option_order(self, question: Question, vote_index: int) -> list[Any]:
        options = list(question.options)
        if self.shuffle_options:
            random.Random(self.base_seed + vote_index).shuffle(options)
        return options

    def _metadata(self, votes: list[AnswerPrediction], judge_raw_text: str | None, method: str) -> dict[str, Any]:
        return {
            "strategy": self.name,
            "model_name": getattr(self.llm, "model_name", "unknown"),
            "judge_model_name": getattr(self.judge_llm, "model_name", "unknown"),
            "device": getattr(self.llm, "device_summary", "unknown"),
            "decision_method": method,
            "judge_scope": self.judge_scope,
            "rejected_judge_fallback": self.rejected_judge_fallback,
            "candidate_devices": [
                getattr(candidate_llm, "device_summary", "unknown") for candidate_llm in self.candidate_llms
            ],
            "judge_device": getattr(self.judge_llm, "device_summary", "unknown"),
            "votes": [
                {
                    "option_id": vote.option_id,
                    "confidence": vote.confidence,
                    "reasoning": vote.reasoning,
                    "raw_text": str(vote.metadata.get("raw_text", ""))[:300],
                    "model_name": vote.metadata.get("model_name"),
                    "sample_seed": vote.metadata.get("sample_seed"),
                    "option_order": vote.metadata.get("option_order"),
                }
                for vote in votes
            ],
            "judge_raw_text": judge_raw_text,
        }


def build_prompt(question: Question) -> str:
    options = "\n".join(f"{option.id}) {option.text}" for option in question.options)
    return "\n\n".join(
        [
            "Pick the best answer. Return only the option id number.",
            f"Q: {question.text}",
            options,
            "Answer:",
        ]
    )


def build_qwen_prompt(question: Question) -> str:
    options = "\n".join(f"{option.id}) {option.text}" for option in question.options)
    return "\n\n".join(
        [
            "Choose the best answer to this multiple-choice question.",
            f"Q: {question.text}",
            options,
            "After thinking, finish with exactly: option_id: <number>",
        ]
    )


def build_council_vote_prompt(question: Question, option_order: list[Any] | None = None) -> str:
    options = "\n".join(f"{option.id}) {option.text}" for option in (option_order or question.options))
    return "\n\n".join(
        [
            "Pick the best answer. Check words such as NOT and EXCEPT.",
            "Return JSON with keys option_id, confidence, and reason. Use the listed numeric ID. Keep the reason short.",
            f"Q: {question.text}",
            options,
        ]
    )


def build_judge_prompt(
    question: Question,
    votes: list[AnswerPrediction],
    judge_scope: str = "candidate_only",
) -> str:
    options = "\n".join(f"{option.id}) {option.text}" for option in question.options)
    summaries = "\n".join(
        f"vote {index + 1}: option={vote.option_id}, confidence={vote.confidence}, reason={vote.reasoning or ''}"
        for index, vote in enumerate(votes)
    )
    supported = ", ".join(str(option_id) for option_id in sorted({vote.option_id for vote in votes}))
    scope_text = (
        f"Choose only one proposed option id: {supported}."
        if judge_scope == "candidate_only"
        else "You may choose any listed option."
    )
    return "\n\n".join(
        [
            f"Choose the final answer from the candidates. {scope_text} Return only the option id number.",
            f"Q: {question.text}",
            options,
            summaries,
            "Answer:",
        ]
    )


def parse_answer_prediction(raw_text: str, question: Question, strategy_name: str = "llm") -> AnswerPrediction:
    payload = _parse_payload(raw_text)
    option_id = _coerce_int(payload.get("option_id"))
    if option_id not in question.valid_option_ids():
        option_id = _match_option_text(raw_text, question)

    fallback = option_id not in question.valid_option_ids()
    if fallback:
        option_id = question.first_option().id

    option = question.require_option(option_id)
    reason = payload.get("reason")
    if reason is not None:
        reason = str(reason).strip()[:300]

    return AnswerPrediction(
        option_id=option.id,
        answer_text=option.text,
        confidence=_clamp_confidence(payload.get("confidence")),
        reasoning=reason,
        metadata={
            "strategy": strategy_name,
            "raw_text": raw_text,
            "fallback": fallback,
            "parsed_payload": payload,
        },
    )


def _majority_option(votes: list[AnswerPrediction]) -> int | None:
    counts: dict[int, int] = {}
    for vote in votes:
        counts[vote.option_id] = counts.get(vote.option_id, 0) + 1
    option_id, count = max(counts.items(), key=lambda item: (item[1], -item[0]))
    return option_id if count > len(votes) / 2 else None


def _selected_vote_prediction(
    question: Question,
    votes: list[AnswerPrediction],
    option_id: int,
) -> AnswerPrediction:
    supporters = [vote for vote in votes if vote.option_id == option_id]
    confidence_values = [vote.confidence for vote in supporters if vote.confidence is not None]
    option = question.require_option(option_id)
    return AnswerPrediction(
        option_id=option.id,
        answer_text=option.text,
        confidence=sum(confidence_values) / len(confidence_values) if confidence_values else None,
        reasoning="Council majority selected this answer.",
        metadata={"strategy": "council", "fallback": False},
    )


def _weighted_vote(question: Question, votes: list[AnswerPrediction]) -> AnswerPrediction:
    scores: dict[int, float] = {}
    counts: dict[int, int] = {}
    for vote in votes:
        weight = vote.confidence if vote.confidence is not None else 0.5
        scores[vote.option_id] = scores.get(vote.option_id, 0.0) + weight
        counts[vote.option_id] = counts.get(vote.option_id, 0) + 1
    option_id = max(scores, key=lambda item: (counts[item], scores[item], -item))
    option = question.require_option(option_id)
    total_score = sum(scores.values())
    return AnswerPrediction(
        option_id=option.id,
        answer_text=option.text,
        confidence=scores[option_id] / total_score if total_score else None,
        reasoning="Confidence-weighted council vote.",
        metadata={"strategy": "council", "fallback": False},
    )


def _council_fallback(question: Question, reason: str, votes: list[AnswerPrediction]) -> AnswerPrediction:
    option = question.first_option()
    return AnswerPrediction(
        option_id=option.id,
        answer_text=option.text,
        confidence=0.0,
        reasoning=reason,
        metadata={
            "strategy": "council",
            "fallback": True,
            "decision_method": "fallback",
            "votes": votes,
        },
    )


def _parse_payload(raw_text: str) -> dict[str, Any]:
    text = raw_text.strip()
    for candidate in [text, *_json_blocks(text)]:
        try:
            data = json.loads(candidate)
            if isinstance(data, dict):
                return data
        except json.JSONDecodeError:
            pass

    payload: dict[str, Any] = {}
    bare_option_match = re.fullmatch(r"\s*(-?\d+)\s*[\.\)]?\s*", text)
    if bare_option_match:
        payload["option_id"] = bare_option_match.group(1)
        return payload
    option_match = re.search(r"(?:option_id|option|answer)\D+(-?\d+)", text, flags=re.IGNORECASE)
    if option_match:
        payload["option_id"] = option_match.group(1)
    confidence_match = re.search(r"confidence\D+([01](?:\.\d+)?)", text, flags=re.IGNORECASE)
    if confidence_match:
        payload["confidence"] = confidence_match.group(1)
    reason_match = re.search(
        r'["\']?(?:reason|justification)["\']?\s*[:=]\s*["\']?([^"\'\n\r}]*)',
        text,
        flags=re.IGNORECASE,
    )
    if reason_match:
        reason = reason_match.group(1).strip(" ,")
        if reason:
            payload["reason"] = reason
    return payload


def _json_blocks(text: str) -> list[str]:
    return re.findall(r"\{.*?\}", text, flags=re.DOTALL)


def _coerce_int(value: Any) -> int | None:
    try:
        return int(value)
    except (TypeError, ValueError):
        return None


def _clamp_confidence(value: Any) -> float | None:
    if value is None:
        return None
    try:
        return max(0.0, min(1.0, float(value)))
    except (TypeError, ValueError):
        return None


def _match_option_text(raw_text: str, question: Question) -> int | None:
    lowered = raw_text.lower()
    for option in question.options:
        if option.text.lower() in lowered:
            return option.id
    return None


def _tokenize_prompt(processor: Any, prompt: str) -> dict[str, Any]:
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    if hasattr(processor, "apply_chat_template"):
        try:
            return processor.apply_chat_template(
                messages,
                add_generation_prompt=True,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
            )
        except Exception:
            pass
    return processor(_text_chat_prompt(prompt), return_tensors="pt")


def _text_chat_prompt(prompt: str) -> str:
    return f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n"


def _require_supported_transformers(version: str) -> None:
    if Version(version) < Version("5.7.0"):
        raise RuntimeError(
            "The local Gemma/Qwen models require transformers>=5.7.0 in this notebook kernel. "
            f"Current transformers version is {version}. Run the dependency install cell "
            "with INSTALL_DEPS=True, then restart the kernel and rerun setup."
        )


def _quantization_kwargs(quantize_4bit: bool) -> dict[str, Any]:
    if not quantize_4bit:
        return {}
    try:
        import bitsandbytes as _bitsandbytes
        del _bitsandbytes
    except ImportError as exc:
        raise RuntimeError(
            "4-bit quantization needs bitsandbytes in this notebook kernel. "
            "Run `%pip install -U bitsandbytes`, restart the kernel, and rerun setup."
        ) from exc

    import torch
    from transformers import BitsAndBytesConfig

    return {
        "quantization_config": BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
    }


def _words(text: str) -> list[str]:
    return re.findall(r"[A-Za-z0-9]+", text.lower())


@tool
def web_search_tool(query: str) -> str:
    """
    Use this tool ONLY for modern pop culture, movies, music, recent events (up to current year 2026), 
    awards, or current news. Input must be a short, concise web search query string.
    """
    try:
        clean_query = urllib.parse.quote(query.strip())
        url = f"https://api.duckduckgo.com/?q={clean_query}&format=json&no_html=1&skip_disambig=1"
        
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=4) as response:
            data = json.loads(response.read().decode())
            
        abstract = data.get("AbstractText", "")
        if abstract:
            return f"Web Search Context: {abstract}"
            
        related = data.get("RelatedTopics", [])
        if related and "Text" in related[0]:
            return f"Web Search Context: {related[0]['Text']}"
            
        return "No direct web results found. Try simplifying the query."
    except Exception as e:
        return f"Web search failed: {str(e)}"

@tool
def calculator_tool(expression: str) -> float:
    """
    Computes mathematical operations, this tool does.
    An argument it takes, a basic Python math string like '2 + 2' or '1.95 ** 10' it must be.
    """
    clean_expr = re.sub(r'[^0-9\+\-\*\/\(\)\.\s]', '', expression)
    return float(eval(clean_expr, {"__builtins__": None}, {}))

@tool
def wikipedia_tool(search_term: str) -> str:
    """
    Historical facts, biographies or general knowledge, this tool finds.
    A single entity, historical person, or academic concept as search_term string, it requires.
    """
    import urllib.request
    import urllib.parse
    import json
    from bs4 import BeautifulSoup
    try:
        query = urllib.parse.quote(search_term.strip())
        url = f"https://en.wikipedia.org/w/api.php?action=query&list=search&srsearch={query}&format=json"
        req = urllib.request.Request(url, headers={'User-Agent': 'PoliMillionaireAgent/1.0'})
        
        with urllib.request.urlopen(req, timeout=4) as response:
            data = json.loads(response.read().decode())
            
        results = data.get("query", {}).get("search", [])
        if not results:
            return "No information found in Wikipedia."
            
        snippet = BeautifulSoup(results[0].get("snippet", ""), "html.parser").get_text()
        return f"Fact context found: {snippet}"
    except Exception as e:
        return f"Error during search: {str(e)}"



class LangChainAgenticStrategy(BaseStrategy):
    name = "langchain_agent"

    def __init__(self, raw_llm: LocalLLM, fallback_strategy: BaseStrategy | None = None):
        # Wraps your existing LocalLLM (Gemma/Qwen) into a LangChain compatible interface
        self.raw_llm = raw_llm
        self.fallback = fallback_strategy or HeuristicStrategy()
        self.tools = [calculator_tool, wikipedia_tool, web_search_tool]
        self._setup_agent_prompts()

    def _setup_agent_prompts(self):
        # Renderizamos las descripciones dinámicamente incluyendo DuckDuckGo
        rendered_tools = render_text_description(self.tools)
        
        system_routing = f"""
You are an advanced orchestrator for the quiz game 'Who wants to be a PoliMillionaire?'.
Your job is to decide if you need an external tool to answer the question accurately.

Here are the names and descriptions for each tool available:
{rendered_tools}

STRICT SELECTION CRITERIA:
1. If the question requires math, counting, formulas, or arithmetic operations, you MUST choose 'calculator_tool'.
2. If the question is about established history, global geography, science, or older biographies, you MUST choose 'wikipedia_tool'.
3. If the question is about modern pop culture, movies, actors, music, awards, current news, or events up to the current year 2026, you MUST choose 'web_search_tool'.
4. If you DO NOT need a tool to answer the question, you MUST set 'name': "none" and 'arguments': {{"query": ""}}.

The response format MUST be exactly a valid JSON block like this:
{{"name": "tool_name", "arguments": {{"query": "search query text here"}}}}
"""
        self.routing_prompt = ChatPromptTemplate.from_messages([
            ("system", system_routing),
            ("user", "Question: {{input}}\nOptions: {{options_text}}")
        ], template_format="jinja2")

        system_answering = """
You are a brilliant contestant playing 'Who wants to be a PoliMillionaire?'.
Additional verified context from your tools: {{context}}

Analyze the question and select the single best option ID.
You MUST return your response ONLY as a JSON blob matching this structure:
{"option_id": 0, "confidence": 0.95, "reasoning": "Your step-by-step logic here"}
"""
        self.answer_prompt = ChatPromptTemplate.from_messages([
            ("system", system_answering),
            ("user", "Question: {{input}}\nOptions:\n{{options_text}}")
        ], template_format="jinja2")

    

    def _invoke_tool(self, tool_name: str, tool_args: dict) -> str:
        if tool_name == "none" or not tool_name:
            return "No additional context needed."
            
        tool_map = {t.name: t for t in self.tools}
        if tool_name in tool_map:
            try:
                actual_args = tool_args
                if tool_name in ["wikipedia_tool", "web_search_tool"]:
                    extracted_text = tool_args.get("query") or tool_args.get("search_term") or tool_args.get("expression")
                    if not extracted_text and isinstance(tool_args, dict) and tool_args:
                        extracted_text = list(tool_args.values())[0]
                    
                    if tool_name == "wikipedia_tool":
                        actual_args = {"search_term": str(extracted_text or "")}
                    else:
                        actual_args = {"query": str(extracted_text or "")}
                        
                elif tool_name == "calculator_tool":
                    extracted_expr = tool_args.get("expression") or tool_args.get("query")
                    if not extracted_expr and isinstance(tool_args, dict) and tool_args:
                        extracted_expr = list(tool_args.values())[0]
                    actual_args = {"expression": str(extracted_expr or "2+2")}

                result = tool_map[tool_name].invoke(actual_args)
                return str(result)
            except Exception as e:
                return f"Tool execution failed: {str(e)}"
        return "Unknown tool requested."

    def answer(self, question: Question) -> AnswerPrediction:
        try:
            import traceback  # <-- Para ver la línea exacta del error
            options_str = "\n".join([f"ID {o.id}: {o.text}" for o in question.options])
            
            # --- PHASE 1: TOOL ROUTING ---
            formatted_routing_prompt = self.routing_prompt.format(
                input=question.text,
                options_text=options_str
            )
            
            if hasattr(self.raw_llm, 'generate'):
                # Usamos max_time en lugar de generation_max_time_seconds
                routing_output = self.raw_llm.generate(formatted_routing_prompt, max_new_tokens=512, max_time=30.0)
            else:
                routing_output = self.raw_llm.invoke(formatted_routing_prompt)
            
            routing_text = getattr(routing_output, "text_content", str(routing_output))
            
            print(f"\n--- [DEBUG AGENTE] Fase 1: Respuesta de Enrutamiento cruda ---\n{routing_text}\n-------------------------------------------------")
            
            try:
                match_routing = re.search(r'\{.*\}', routing_text, re.DOTALL)
                if match_routing:
                    routing_json = json.loads(match_routing.group(0))
                    tool_name = routing_json.get("name", "none")
                    tool_arguments = routing_json.get("arguments", {})
            except Exception as json_err:
                print(f"❌ [DEBUG AGENTE] Error al parsear JSON de herramientas: {json_err}")
            
            # --- PHASE 2: TOOL EXECUTION ---
            context_data = self._invoke_tool(tool_name, tool_arguments)
            # 🔍 [DEBUG 2]: Ver si la herramienta arrojó datos útiles o falló
            print(f"ℹ️ [DEBUG AGENTE] Fase 2: Herramienta elegida [{tool_name}] | Contexto obtenido: {context_data[:100]}...")
            
            # --- PHASE 3: FINAL ANSWER GENERATION ---
            formatted_answer_prompt = self.answer_prompt.format(
                context=context_data,
                input=question.text,
                options_text=options_str
            )
            
            if hasattr(self.raw_llm, 'generate'):
                # Usamos max_time en lugar de generation_max_time_seconds
                final_output = self.raw_llm.generate(formatted_answer_prompt, max_new_tokens=512, max_time=30.0)
            else:
                final_output = self.raw_llm.invoke(formatted_answer_prompt)
                
            final_text = getattr(final_output, "text_content", str(final_output))
            
            # 🔍 [DEBUG 3]: Ver qué responde el modelo como respuesta final antes del parseo
            print(f"\n--- [DEBUG AGENTE] Fase 3: Respuesta final cruda del modelo ---\n{final_text}\n-------------------------------------------------")
            
            # --- PHASE 4: ULTRA-ROBUST PARSING ---
            prediction_data = self._robust_parse_final_answer(final_text, question)
            if prediction_data:
                prediction_data.metadata.update({
                    "strategy": self.name,
                    "used_tool": tool_name,
                    "tool_arguments": str(tool_arguments),
                    "tool_output": context_data,
                    "fallback": False
                })
                return prediction_data
            else:
                print("❌ [DEBUG AGENTE] Fase 4: '_robust_parse_final_answer' devolvió None. El texto no contenía un formato procesable.")
                
        except Exception as e:
            # 🚨 Imprime la línea exacta del código de la estrategia que crasheó
            print(f"\n💥 [DEBUG AGENTE] CRASH INTERNO EN LA ESTRATEGIA:")
            traceback.print_exc()
            
        # Safe recovery si algo falló arriba
        fallback_pred = self.fallback.answer(question)
        fallback_pred.metadata["fallback"] = True
        fallback_pred.metadata["fallback_reason"] = "Pipeline exception occurred"
        return fallback_pred

    def _robust_parse_final_answer(self, text: str, question: Question) -> AnswerPrediction | None:
        # Regex to locate any JSON block inside the model output, we deploy!
        try:
            match = re.search(r'\{.*\}', text, re.DOTALL)
            if match:
                clean_json_str = match.group(0)
                data = json.loads(clean_json_str)
                
                # Extract the option ID defensively
                raw_id = data.get("option_id")
                if raw_id is not None:
                    selected_id = int(raw_id)
                    chosen_option = next((o for o in question.options if o.id == selected_id), question.options[0])
                    
                    return AnswerPrediction(
                        option_id=chosen_option.id,
                        answer_text=chosen_option.text,
                        confidence=float(data.get("confidence", 0.85)),
                        reasoning=str(data.get("reasoning", "Parsed via robust JSON engine.")),
                        metadata={}
                    )
        except Exception as parse_exc:
            print(f"[{self.name}] Regex JSON parser missed, fallback to heuristic extraction: {parse_exc}")

        # Emergency Fallback: If JSON is completely corrupted, search for integers in text
        # (Very common for 4-bit models under low max_new_tokens)
        for token in text.split():
            clean_token = re.sub(r'[^0-9]', '', token)
            if clean_token.isdigit():
                val = int(clean_token)
                for opt in question.options:
                    if opt.id == val:
                        return AnswerPrediction(
                            option_id=opt.id,
                            answer_text=opt.text,
                            confidence=0.5,
                            reasoning=f"Extracted direct integer reference '{val}' from corrupted output.",
                            metadata={}
                        )
        return None


## runner.py

In [47]:
from __future__ import annotations

"""Game runner, JSONL logs, and small result summaries."""

import json
import time
from concurrent.futures import ThreadPoolExecutor, TimeoutError
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any


def from_client_question(client_question: Any) -> Question:
    return Question(
        id=int(getattr(client_question, "id")),
        text=str(getattr(client_question, "text")),
        options=[
            AnswerOption(id=int(getattr(option, "id")), text=str(getattr(option, "text")))
            for option in getattr(client_question, "options")
        ],
        level=int(getattr(client_question, "level", 0) or 0),
        metadata={"source": "millionaire_client"},
    )


class SafeDelay:
    def __init__(self, seconds: float = 1.0):
        self.seconds = max(0.0, seconds)
        self._last_call: float | None = None

    def wait(self) -> None:
        now = time.monotonic()
        if self._last_call is not None:
            remaining = self.seconds - (now - self._last_call)
            if remaining > 0:
                time.sleep(remaining)
        self._last_call = time.monotonic()


class GameRunner:
    def __init__(
        self,
        client: Any,
        safe_delay_seconds: float = 1.0,
        answer_timeout_seconds: float = 25.0,
        logger: "RunLogger | None" = None,
    ):
        self.client = client
        self.safe_delay = SafeDelay(safe_delay_seconds)
        self.answer_timeout_seconds = answer_timeout_seconds
        self.logger = logger

    def play(self, competition_id: int, strategy: BaseStrategy) -> Any:
        self.safe_delay.wait()
        game = self.client.game.start(competition_id=competition_id)
        while game.in_progress:
            if game.current_question is None:
                break
            question = from_client_question(game.current_question)
            started_at = time.monotonic()
            prediction = self._safe_answer(strategy, question)
            elapsed = time.monotonic() - started_at
            self.safe_delay.wait()
            try:
                result = game.answer(prediction.option_id)
            except Exception as exc:
                result = SubmissionErrorResult(error=str(exc))
                self._log(question, prediction, result, elapsed, strategy)
                break
            self._log(question, prediction, result, elapsed, strategy)
            if getattr(result, "game_over", False):
                break
        return game

    def _safe_answer(self, strategy: BaseStrategy, question: Question) -> AnswerPrediction:
        fallback = _fallback_prediction(question, "Strategy failed or timed out.")
        executor = ThreadPoolExecutor(max_workers=1)
        try:
            future = executor.submit(strategy.answer, question)
            prediction = future.result(timeout=self.answer_timeout_seconds)
        except TimeoutError:
            executor.shutdown(wait=False, cancel_futures=True)
            fallback.metadata["error"] = f"Timed out after {self.answer_timeout_seconds}s"
            return fallback
        except Exception as exc:
            executor.shutdown(wait=False, cancel_futures=True)
            fallback.metadata["error"] = str(exc)
            return fallback
        executor.shutdown(wait=False)
        if prediction.option_id not in question.valid_option_ids():
            fallback.metadata["error"] = "Strategy returned invalid option_id"
            fallback.metadata["original_prediction"] = prediction.metadata
            return fallback
        return prediction

    def _log(
        self,
        question: Question,
        prediction: AnswerPrediction,
        result: Any,
        elapsed_seconds: float,
        strategy: BaseStrategy,
    ) -> None:
        if self.logger:
            self.logger.log_attempt(
                question=question,
                prediction=prediction,
                result=result,
                elapsed_seconds=elapsed_seconds,
                strategy_name=getattr(strategy, "name", strategy.__class__.__name__),
            )


class RunLogger:
    def __init__(self, path: str | Path):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)

    def log_attempt(
        self,
        question: Question,
        prediction: AnswerPrediction,
        result: Any,
        elapsed_seconds: float,
        strategy_name: str,
    ) -> None:
        row = {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "strategy_name": strategy_name,
            "question": asdict(question),
            "prediction": asdict(prediction),
            "elapsed_seconds": elapsed_seconds,
            "result": {
                "correct": getattr(result, "correct", None),
                "game_over": getattr(result, "game_over", None),
                "earned_amount": getattr(result, "earned_amount", None),
                "timed_out": getattr(result, "timed_out", None),
                "status": getattr(result, "status", None),
                "current_level": getattr(result, "current_level", None),
                "error": getattr(result, "error", None),
            },
        }
        with self.path.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")


def load_jsonl(path: str | Path) -> list[dict[str, Any]]:
    with Path(path).open("r", encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def summarize_attempts(rows: list[dict[str, Any]]) -> dict[str, Any]:
    total = len(rows)
    correct = sum(1 for row in rows if row.get("result", {}).get("correct") is True)
    timed_out = sum(1 for row in rows if row.get("result", {}).get("timed_out") is True)
    elapsed_values = [row.get("elapsed_seconds", 0.0) for row in rows]
    return {
        "total": total,
        "correct": correct,
        "accuracy": correct / total if total else 0.0,
        "timed_out": timed_out,
        "avg_elapsed_seconds": sum(elapsed_values) / total if total else 0.0,
    }


def _fallback_prediction(question: Question, reason: str) -> AnswerPrediction:
    option = question.first_option()
    return AnswerPrediction(
        option_id=option.id,
        answer_text=option.text,
        confidence=0.0,
        reasoning=reason,
        metadata={"fallback": True},
    )


@dataclass
class SubmissionErrorResult:
    correct: bool | None = None
    game_over: bool = True
    earned_amount: float | None = None
    timed_out: bool = False
    status: str = "submission_error"
    current_level: int | None = None
    error: str = ""


# Notebook game

In [48]:
RUN_API_CHECK = False
RUN_LIVE_GAME = True
RUN_OFFLINE_BENCHMARK = False
PROMPT_FOR_CREDENTIALS = False

API_URL = "http://131.175.15.22:51111/"
COMPETITION_ID = 0
SAFE_DELAY_SECONDS = 2.0
ANSWER_TIMEOUT_SECONDS = 25.0

# Cambia a True el parámetro "run" únicamente del modelo que vas a testear hoy
MODELS_TO_RUN = [
    {"label": "Gemma 4 E2B", "kind": "gemma", "model_id": "google/gemma-4-E2B-it", "run": False},
    {"label": "Gemma 4 E4B", "kind": "gemma", "model_id": "google/gemma-4-E4B-it", "run": False},
    {"label": "Qwen3.5 2B Thinking", "kind": "qwen_thinking", "model_id": "Qwen/Qwen3.5-2B", "run": False},
    {"label": "Gemma 4 E2B Council", "kind": "gemma_council", "model_id": "google/gemma-4-E2B-it", "votes": 3, "run": False},
    {"label": "Gemma + Qwen Mixed Council", "kind": "mixed_council", "model_id": "google/gemma-4-E2B-it + Qwen/Qwen3.5-2B", "run": False},
    {"label": "Gemma 4 E2B (4-bit)", "kind": "gemma_quantized", "model_id": "google/gemma-4-E2B-it", "run": False},
    {"label": "Gemma + Qwen Mixed Council (4-bit)", "kind": "mixed_quantized", "model_id": "google/gemma-4-E2B-it + Qwen/Qwen3.5-2B", "run": False},
    {"label": "Gemma 4 E2B LangChain Agent (4-bit)", "kind": "langchain_agentic", "model_id": "google/gemma-4-E2B-it", "run": True},
]

print("API check:", RUN_API_CHECK, "| Live game:", RUN_LIVE_GAME, "| Benchmark:", RUN_OFFLINE_BENCHMARK)
for item in MODELS_TO_RUN:
    print("-", item["label"], "run=", item["run"])

API check: False | Live game: True | Benchmark: False
- Gemma 4 E2B run= False
- Gemma 4 E4B run= False
- Qwen3.5 2B Thinking run= False
- Gemma 4 E2B Council run= False
- Gemma + Qwen Mixed Council run= False
- Gemma 4 E2B (4-bit) run= False
- Gemma + Qwen Mixed Council (4-bit) run= False
- Gemma 4 E2B LangChain Agent (4-bit) run= True


In [49]:
from pathlib import Path
import os
from kaggle_secrets import UserSecretsClient

# En Kaggle, la carpeta de escritura permitida es /kaggle/working
OUTPUT_DIR = Path("/kaggle/working")

rey_user = UserSecretsClient().get_secret("rey_user")
rey_password = UserSecretsClient().get_secret("rey_password")
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

client = MillionaireClient("http://131.175.15.22:51111")

In [50]:
import gc
import time
import importlib.util
from datetime import datetime, timezone

warmup_question = Question(1, "What is 2 + 2?", [AnswerOption(0, "3"), AnswerOption(1, "4"), AnswerOption(2, "5"), AnswerOption(3, "22")])
BENCHMARK_SET = [
    (warmup_question, 1),
    (Question(2, "Which song was NOT written by Bob Dylan?", [AnswerOption(0, "Like a Rolling Stone"), AnswerOption(1, "Blowin' in the Wind"), AnswerOption(2, "The Times They Are A-Changin'"), AnswerOption(3, "Hound Dog")]), 3),
    (Question(3, "What was Whitney Houston's debut album?", [AnswerOption(0, "Whitney Houston"), AnswerOption(1, "Just Whitney"), AnswerOption(2, "I'm Your Baby Tonight"), AnswerOption(3, "Whitney")]), 0),
]

def clear_model_memory():
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception as exc:
        print("Limpieza de memoria:", exc)

def mixed_council(quantized=False):
    if quantized and importlib.util.find_spec("bitsandbytes") is None:
        raise RuntimeError("Para 4-bit, ejecuta `%pip install -U bitsandbytes` y reinicia el kernel.")
    gemma = GemmaLLM(GemmaLLMConfig(model_id="google/gemma-4-E2B-it", quantize_4bit=quantized, max_new_tokens=32, do_sample=True, seed=42, generation_max_time_seconds=6.0, timeout_seconds=120.0))
    qwen = QwenLLM(QwenLLMConfig(model_id="Qwen/Qwen3.5-2B", quantize_4bit=quantized, max_new_tokens=32, temperature=0.7, top_p=0.8, top_k=20, do_sample=True, seed=43, enable_thinking=False, generation_max_time_seconds=6.0, timeout_seconds=120.0))
    return CouncilStrategy(candidate_llms=[gemma, qwen], judge_llm=gemma, judge_scope="candidate_only" if quantized else "any_option", rejected_judge_fallback="primary_candidate" if quantized else "confidence_weighted", base_seed=200, candidate_max_new_tokens=32, judge_max_new_tokens=8, max_time_per_call=6.0)


def make_strategy(item):
    if item["kind"] == "gemma_quantized":
        if importlib.util.find_spec("bitsandbytes") is None:
            raise RuntimeError("For 4-bit Gemma, run `%pip install -U bitsandbytes`, restart the kernel, and rerun setup.")
        return GemmaStrategy(model_config=GemmaLLMConfig(model_id=item["model_id"], quantize_4bit=True, max_new_tokens=8, temperature=0.0, do_sample=False, num_beams=1, seed=42, generation_max_time_seconds=18.0, timeout_seconds=120.0))
    if item["kind"] == "mixed_quantized":
        return mixed_council(quantized=True)
    if item["kind"] == "langchain_agentic":
        if importlib.util.find_spec("bitsandbytes") is None:
            raise RuntimeError("For 4-bit LangChain Agent, run `%pip install -U bitsandbytes`.")
        
        agent_config = GemmaLLMConfig(
            model_id=item["model_id"], 
            quantize_4bit=True, 
            max_new_tokens=128,
            temperature=0.0, 
            do_sample=False, 
            generation_max_time_seconds=30.0, 
            timeout_seconds=120.0
        )
        base_llm = GemmaLLM(config=agent_config)
        return LangChainAgenticStrategy(raw_llm=base_llm, fallback_strategy=HeuristicStrategy())
    if item["kind"] == "mixed_council":
        return mixed_council()
    if item["kind"] == "gemma_council":
        llm = GemmaLLM(GemmaLLMConfig(model_id=item["model_id"], max_new_tokens=48, do_sample=True, seed=42, generation_max_time_seconds=4.5, timeout_seconds=120.0))
        return CouncilStrategy(llm=llm, num_votes=item.get("votes", 3), base_seed=100, candidate_max_new_tokens=48, judge_max_new_tokens=8, max_time_per_call=4.5)
    if item["kind"] == "qwen_thinking":
        return QwenStrategy(model_config=QwenLLMConfig(model_id=item["model_id"], max_new_tokens=128, temperature=1.0, top_p=0.95, top_k=20, do_sample=True, seed=42, enable_thinking=True, generation_max_time_seconds=18.0, timeout_seconds=120.0))
    return GemmaStrategy(model_config=GemmaLLMConfig(model_id=item["model_id"], max_new_tokens=8, temperature=0.0, do_sample=False, num_beams=1, seed=42, generation_max_time_seconds=18.0, timeout_seconds=120.0))


def clean_name(label):
    return "_".join(label.lower().replace("+", " ").replace("(", " ").replace(")", " ").split())

benchmark_results = []

def benchmark(strategy, label):
    rows = []
    for question, gold_id in BENCHMARK_SET:
        started = time.monotonic()
        prediction = strategy.answer(question)
        seconds = time.monotonic() - started
        correct = prediction.option_id == gold_id
        rows.append((correct, seconds, bool(prediction.metadata.get("fallback"))))
        gold = question.require_option(gold_id)
        print(f"\nQ: {question.text} | Pred: {prediction.option_id} | Correcto: {correct} | Tiempo: {round(seconds, 2)}s")
    accuracy = sum(correct for correct, _, _ in rows) / len(rows)
    summary = {"model": label, "accuracy": accuracy, "avg_seconds": round(sum(seconds for _, seconds, _ in rows) / len(rows), 2), "fallbacks": sum(fallback for _, _, fallback in rows)}
    benchmark_results.append(summary)
    return accuracy >= 0.60 and not any(fallback for _, _, fallback in rows)

# =====================================================================
# DIRECT AUTHENTICATION CONTROL (LOGIN FIX)
# =====================================================================
if RUN_LIVE_GAME:
    # Verify that the required credentials exist within Kaggle Secrets
    if not rey_user or not rey_password:
        raise RuntimeError("Missing 'rey_user' or 'rey_password' credentials in Kaggle Secrets.")
    
    # Force explicit client initialization and login to prevent unauthenticated requests
    print(f"Authenticating with the server for the live game... User: {rey_user}")
    client = MillionaireClient(API_URL)
    client.login(rey_user, rey_password)
    print("Session successfully initiated on PoliMillionaire!")
# =====================================================================

run_results = []
for item in MODELS_TO_RUN:
    if not item.get("run", False):
        continue
    strategy = None
    try:
        print(f"\n=== Iniciando Evaluación: {item['label']} ===")
        strategy = make_strategy(item)
        started = time.monotonic()
        warmup = strategy.answer(warmup_question)
        
        # --- BLOQUE DE DIAGNÓSTICO DE EMERGENCIA ---
        print(f"--> [DIAGNÓSTICO] Warmup Option ID predicho: {warmup.option_id} (Esperado: 1)")
        print(f"--> [DIAGNÓSTICO] ¿Se usó fallback heurístico?: {warmup.metadata.get('fallback')}")
        print(f"--> [DIAGNÓSTICO] Texto de razonamiento del modelo: {getattr(warmup, 'reasoning', 'Ninguno')}")
        # ---------------------------------------------
        
        if warmup.metadata.get("fallback") or warmup.option_id != 1:
            raise RuntimeError("Warmup corrupto o fallido. Bloqueada ejecución en vivo.")
            
        if item["kind"] in {"gemma_quantized", "mixed_quantized"}:
            devices = [model.device_summary for model in strategy.candidate_llms] if isinstance(strategy, CouncilStrategy) else [strategy.llm.device_summary]
            if any(token in " ".join(devices).lower() for token in ("'cpu'", "'disk'", "meta")):
                raise RuntimeError("Modelo desbordado a CPU/Disco. Abortando partida.")
            
            started = time.monotonic()
            speed_check = strategy.answer(warmup_question)
            if (time.monotonic() - started) > 20.0:
                raise RuntimeError("El modelo responde demasiado lento (>20s). Abortando.")

        benchmark_ok = benchmark(strategy, item["label"]) if RUN_OFFLINE_BENCHMARK else True
        if not benchmark_ok and RUN_LIVE_GAME:
            raise RuntimeError("Guardia de benchmark fallida. No se inicia partida real.")

        result = {"label": item["label"], "benchmark_ok": benchmark_ok}
        if RUN_LIVE_GAME:
            run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
            # Cambiado REPO_ROOT por OUTPUT_DIR
            log_path = OUTPUT_DIR / "results" / "runs" / f"{clean_name(item['label'])}_{run_id}.jsonl"
            game = GameRunner(client, SAFE_DELAY_SECONDS, ANSWER_TIMEOUT_SECONDS, RunLogger(log_path)).play(COMPETITION_ID, strategy)
            result.update({"earned_amount": game.earned_amount, "log_path": str(log_path)})
            print(f"Partida finalizada. Puntuación: {game.earned_amount}")
        run_results.append(result)
    finally:
        del strategy
        clear_model_memory()

run_results

Authenticating with the server for the live game... User: reyci
Session successfully initiated on PoliMillionaire!

=== Iniciando Evaluación: Gemma 4 E2B LangChain Agent (4-bit) ===


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]


--- [DEBUG AGENTE] Fase 1: Respuesta de Enrutamiento cruda ---
```json
{"name": "calculator_tool", "arguments": {"expression": "2 + 2"}}
```
-------------------------------------------------
ℹ️ [DEBUG AGENTE] Fase 2: Herramienta elegida [calculator_tool] | Contexto obtenido: 4.0...

--- [DEBUG AGENTE] Fase 3: Respuesta final cruda del modelo ---
```json
{
"option_id": 1,
"confidence": 0.99,
"reasoning": "The question asks for the result of the simple arithmetic operation 2 + 2. The correct mathematical answer is 4. Option ID 1 is 4. Other options (3, 5, 22) are incorrect. Option ID 3 (22) is likely a distractor related to the game's theme, not the mathematical answer."
}
```
-------------------------------------------------
--> [DIAGNÓSTICO] Warmup Option ID predicho: 1 (Esperado: 1)
--> [DIAGNÓSTICO] ¿Se usó fallback heurístico?: False
--> [DIAGNÓSTICO] Texto de razonamiento del modelo: The question asks for the result of the simple arithmetic operation 2 + 2. The correct mathematica

[{'label': 'Gemma 4 E2B LangChain Agent (4-bit)',
  'benchmark_ok': True,
  'earned_amount': 1,
  'log_path': '/kaggle/working/results/runs/gemma_4_e2b_langchain_agent_4-bit_20260527_161048.jsonl'}]

In [51]:
# Cambiado REPO_ROOT por OUTPUT_DIR
run_logs = sorted((OUTPUT_DIR / "results" / "runs").glob("*.jsonl"), key=lambda path: path.stat().st_mtime)
print("Últimos registros en disco:")
for path in run_logs[-3:]:
    print(f"- {path.name}")

if run_logs:
    print(f"\nEstadísticas del último juego ({run_logs[-1].name}):")
    print(summarize_attempts(load_jsonl(run_logs[-1])))

Últimos registros en disco:
